In [6]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
if not (project_root / "api").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Ensure the project root and parent are on PYTHONPATH for Jupyter notebooks.
for candidate in (project_root, project_root.parent):
    if candidate.exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))
        
try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

env_path = Path(project_root) / ".env"
if load_dotenv is not None and env_path.exists():
    load_dotenv(dotenv_path=env_path)

In [8]:
from sqlalchemy.orm import selectinload
from sqlalchemy import select

from core.db import SessionLocal
from models.user import User

In [25]:
from schemas.user import UserCreate
from models.user import Role
from services.crud.user import create_user, get_user_by_id


user_id = "dc76ab12-68bf-4260-aa41-d4164fcdbfbd"


async with SessionLocal() as db:
    result = await db.execute(select(Role).where(Role.name == "admin"))
    role = result.scalar_one_or_none()
    user_in = UserCreate(username='admin3', email=f"admin3@example.com", password='8486')
    user = await create_user(user_in, db)
    await db.refresh(user, ["roles"])
    user.roles.append(role)
    db.add(user)
    await db.commit()
    print("TESTETANDO", user.id)

2026-08-21 16:29:55,885 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-08-21 16:29:55,887 INFO sqlalchemy.engine.Engine SELECT role.id, role.name 
FROM role 
WHERE role.name = %s
2026-08-21 16:29:55,887 INFO sqlalchemy.engine.Engine [cached since 514.2s ago] ('admin',)
2026-08-21 16:29:55,962 INFO sqlalchemy.engine.Engine INSERT INTO user (id, username, email, password_hash, is_active) VALUES (%s, %s, %s, %s, %s)
2026-08-21 16:29:55,963 INFO sqlalchemy.engine.Engine [cached since 347.5s ago] (UUID('b4e4767b-d594-4d55-9845-8ca80f875381'), 'admin3', 'admin3@example.com', '$argon2id$v=19$m=65536,t=3,p=4$PAOaMFnR32uszBoX63YuAg$W9yrxV/i++tS6cJnJOsMYqZLV4MFp2rAbG8ubeDwknw', 1)
2026-08-21 16:29:55,965 INFO sqlalchemy.engine.Engine COMMIT
2026-08-21 16:29:55,969 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-08-21 16:29:55,969 INFO sqlalchemy.engine.Engine SELECT user.id, user.username, user.email, user.password_hash, user.is_active 
FROM user 
WHERE user.id = %s
2026-08-21 16:29:55,

In [21]:
user.roles.append(role)

DetachedInstanceError: Parent instance <User at 0x7badb84f39d0> is not bound to a Session; lazy load operation of attribute 'roles' cannot proceed (Background on this error at: https://sqlalche.me/e/20/bhk3)